# Inline HTML dashboard using notebook MIME output

This notebook returns a Python object with an HTML representation as the final cell result.

In [ ]:
import html
from datetime import datetime, timezone

CATALOG = 'aidp_poc'

summary = spark.sql(f'''SELECT COUNT(*) rows, COUNT(DISTINCT meter_id) meters,
  ROUND(SUM(consumption_kwh),2) kwh, ROUND(MAX(demand_kw),2) demand,
  SUM(CASE WHEN tamper_flag THEN 1 ELSE 0 END) tamper
  FROM {CATALOG}.silver.interval_reading''').first()
trend = spark.sql(f'''SELECT interval_start_utc, ROUND(SUM(consumption_kwh),2) kwh
  FROM {CATALOG}.silver.interval_reading
  WHERE interval_start_utc >= current_timestamp() - INTERVAL 24 HOURS
  GROUP BY interval_start_utc ORDER BY interval_start_utc''').collect()

values = [float(r.kwh or 0) for r in trend]
maximum = max(values) if values else 1
bars = ''.join(f'<div class="bar" style="height:{max(5,100*v/maximum):.1f}%"></div>' for v in values)
stamp = datetime.now(timezone.utc).strftime('%H:%M:%S UTC')

page = f'''<style>
.gp{{font-family:Arial,sans-serif;color:#eef6ff;background:radial-gradient(circle at 10% 0,#2b478f,#111831 50%,#070a13);padding:24px;border-radius:18px}}.head{{display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:12px}}.title{{font-size:29px;font-weight:800}}.live{{background:#164b3e;color:#8dffda;border-radius:20px;padding:8px 14px;font-size:12px;font-weight:bold}}.dot{{display:inline-block;width:9px;height:9px;background:#62ffc4;border-radius:50%;margin-right:7px;animation:p 1s infinite}}.grid{{display:grid;grid-template-columns:repeat(4,minmax(140px,1fr));gap:13px;margin:20px 0}}.card{{background:linear-gradient(135deg,#273d7e,#18214a);border:1px solid #526aaa;border-radius:14px;padding:16px}}.label{{font-size:11px;color:#b7c9f5;letter-spacing:1px}}.value{{font-size:28px;color:#7feaff;font-weight:800;margin-top:7px}}.chart{{height:220px;display:flex;align-items:end;gap:3px;padding:10px 2px;border-bottom:1px solid #6177aa}}.bar{{flex:1;min-width:3px;background:linear-gradient(#84edff,#6473ff);border-radius:5px 5px 0 0;animation:g .8s ease-out}}.signal{{margin-top:16px;background:#7feaff16;border-left:4px solid #7feaff;padding:12px}}@keyframes p{{50%{{transform:scale(1.8);opacity:.3}}}}@keyframes g{{from{{transform:scaleY(.1);transform-origin:bottom}}}}@media(max-width:700px){{.grid{{grid-template-columns:repeat(2,1fr)}}}}</style><div class="gp"><div class="head"><div><div class="title">⚡ GridPulse Live</div><div style="color:#b7c9f5;font-size:12px">AIDP streaming signal · {stamp}</div></div><div class="live"><span class="dot"></span>LIVE SNAPSHOT</div></div><div class="grid"><div class="card"><div class="label">SILVER READINGS</div><div class="value">{summary.rows:,}</div></div><div class="card"><div class="label">ACTIVE METERS</div><div class="value">{summary.meters:,}</div></div><div class="card"><div class="label">TOTAL ENERGY</div><div class="value">{float(summary.kwh or 0):,.0f}</div></div><div class="card"><div class="label">PEAK DEMAND</div><div class="value">{float(summary.demand or 0):,.1f}</div></div></div><b>24-HOUR CONSUMPTION WAVE</b><div class="chart">{bars}</div><div class="signal">Latest tamper signals: <b>{int(summary.tamper or 0)}</b> · Rerun this cell for a fresh live snapshot.</div></div>'''

class InlineDashboard:
    def __init__(self, markup): self.markup = markup
    def _repr_html_(self): return self.markup
    def __repr__(self): return 'GridPulse dashboard - HTML renderer unavailable in this notebook frontend.'

InlineDashboard(page)